In [3]:
!pip install -q sentence-transformers

In [4]:
# ============================================================
# 0. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm

from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
from transformers import pipeline, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from sentence_transformers import SentenceTransformer, util


# ============================================================
# 1. LOAD DATA
# ============================================================
data = pd.read_csv(
    '/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'
)
print(f"Dataset shape: {data.shape}")


# ============================================================
# 2. LIGHT CLEANING (KEEP BERT CONTEXT)
# ============================================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# ============================================================
# 3. ASPECT DEFINITIONS
# ============================================================
aspect_labels = [
    "Acting performance of actors",
    "Story and plot quality",
    "Cinematography and visuals",
    "Background music and soundtrack",
    "Direction and screenplay",
    "Action and stunts",
    "Emotional impact",
    "Dialogue quality",
    "Pacing and editing",
    "Overall movie experience"
]


# ============================================================
# 4. EMBEDDING MODEL
# ============================================================
print("\nLoading Sentence Transformer...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
aspect_embeddings = embedder.encode(aspect_labels, convert_to_tensor=True)


# ============================================================
# 4b. DISTILBERT SENTENCE-LEVEL LABELER
# ============================================================
print("Loading DistilBERT sentiment labeler...")
labeler = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0,        # GPU
    batch_size=128
)


# ============================================================
# 5. ASPECT EXTRACTION (SEMANTIC SIMILARITY)
# ============================================================
def extract_aspects_semantic(review, threshold=0.45):
    sentences = re.split(r'[.!?\n]+', review)
    sentences = [clean_text(s) for s in sentences if len(s.strip()) > 5]

    if not sentences:
        return {}

    sent_embeddings = embedder.encode(sentences, convert_to_tensor=True)
    results = {}

    for i, aspect in enumerate(aspect_labels):
        sims = util.cos_sim(aspect_embeddings[i], sent_embeddings)[0]
        top_k = torch.topk(sims, k=min(2, len(sims)))

        matched = [
            sentences[idx]
            for score, idx in zip(top_k.values, top_k.indices)
            if score > threshold
        ]

        if matched:
            results[aspect] = ' '.join(matched)

    return results


# ============================================================
# 6. BUILD DATASET (DISTILBERT LABELS — NO WEAK SUPERVISION)
# ============================================================
def build_aspect_dataset(data, sample_size=25000):
    all_texts   = []
    all_aspects = []

    sampled = data.sample(n=min(sample_size, len(data)), random_state=42)

    for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc="Extracting aspects"):
        aspects = extract_aspects_semantic(row['review'])
        for aspect, text in aspects.items():
            if len(text.split()) < 6:    # skip very short noisy segments
                continue
            all_texts.append(text)
            all_aspects.append(aspect)

    # Label each sentence directly with DistilBERT (batched = fast on GPU)
    print(f"\nLabeling {len(all_texts)} sentences with DistilBERT...")
    raw_labels = labeler(all_texts, truncation=True, max_length=256)
    labels = ['positive' if r['label'] == 'POSITIVE' else 'negative'
              for r in raw_labels]

    df = pd.DataFrame({
        'aspect':    all_aspects,
        'text':      all_texts,
        'sentiment': labels
    })
    print(f"\nAspect dataset shape: {df.shape}")
    print(df['sentiment'].value_counts())
    print(df['aspect'].value_counts())
    return df

aspect_data = build_aspect_dataset(data, sample_size=25000)


# ============================================================
# 7. ENCODE LABELS
# ============================================================
label_encoder = LabelEncoder()
aspect_data['label'] = label_encoder.fit_transform(aspect_data['sentiment'])
print(f"\nClasses: {label_encoder.classes_}")


# ============================================================
# 8. DEVICE + CLASS WEIGHTS
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(aspect_data['label']),
    y=aspect_data['label']
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)


# ============================================================
# 9. TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    aspect_data['text'].tolist(),
    aspect_data['label'].tolist(),
    test_size=0.2,
    stratify=aspect_data['label'],
    random_state=42
)
print(f"\nTrain size: {len(X_train)} | Test size: {len(X_test)}")


# ============================================================
# 10. TOKENIZATION
# ============================================================
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=True)

def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=200,       # increased from 100
        return_tensors='pt'
    )

print("\nTokenizing...")
train_enc = tokenize(X_train)
test_enc  = tokenize(X_test)


# ============================================================
# 11. DATA LOADERS
# ============================================================
train_dataset = TensorDataset(
    train_enc['input_ids'],
    train_enc['attention_mask'],
    torch.tensor(y_train)
)
test_dataset = TensorDataset(
    test_enc['input_ids'],
    test_enc['attention_mask'],
    torch.tensor(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32)


# ============================================================
# 12. MODEL
# ============================================================
config = BertConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)


# ============================================================
# 13. OPTIMIZER + SCHEDULER
# ============================================================
EPOCHS        = 5
LEARNING_RATE = 2e-5          # reverted from 1.5e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


# ============================================================
# 14. TRAINING LOOP
# ============================================================
def train_epoch():
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(train_loader, desc="Training"):
        input_ids, mask, labels = [b.to(device) for b in batch]

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=mask)
        logits  = outputs.logits

        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(train_loader), correct / total


# ============================================================
# 15. EVALUATION
# ============================================================
def evaluate():
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids, mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids, attention_mask=mask)
            preds   = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc    = accuracy_score(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds,
        target_names=label_encoder.classes_
    )
    return acc, report


# ============================================================
# 16. TRAIN WITH EARLY STOPPING
# ============================================================
print("\n" + "="*50)
print("TRAINING BERT FOR ASPECT-BASED SENTIMENT ANALYSIS")
print("="*50)

best_acc = 0
patience = 2
counter  = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch()
    val_acc, val_report   = evaluate()

    print(f"  Train Loss : {train_loss:.4f} | Train Acc : {train_acc*100:.2f}%")
    print(f"  Val   Acc  : {val_acc*100:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        counter  = 0
        model.save_pretrained('/kaggle/working/best_model')
        tokenizer.save_pretrained('/kaggle/working/best_model')
        print("  ✓ Best model saved")
    else:
        counter += 1
        print(f"  No improvement ({counter}/{patience})")
        if counter >= patience:
            print("  Early stopping triggered")
            break


# ============================================================
# 17. FINAL EVALUATION (load best checkpoint)
# ============================================================
print("\n" + "="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)

model = BertForSequenceClassification.from_pretrained('/kaggle/working/best_model').to(device)
final_acc, final_report = evaluate()
print(f"Accuracy : {final_acc*100:.2f}%")
print("\nClassification Report:")
print(final_report)


# ============================================================
# 18. INFERENCE
# ============================================================
def predict(review):
    aspects = extract_aspects_semantic(review)
    results = {}

    model.eval()
    with torch.no_grad():
        for aspect, text in aspects.items():
            enc    = tokenizer(text, return_tensors='pt', truncation=True,
                               max_length=200, padding=True).to(device)
            logits = model(**enc).logits
            pred   = torch.argmax(logits, dim=1).item()
            conf   = torch.softmax(logits, dim=1).max().item()

            results[aspect] = {
                'sentiment':  label_encoder.inverse_transform([pred])[0],
                'confidence': f"{conf*100:.1f}%",
                'text_used':  text[:80] + '...'
            }

    return results


# ============================================================
# 19. DEMO
# ============================================================
sample_review = "One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. The acting was absolutely phenomenal, especially the lead actress. The plot was confusing and hard to follow at times. The cinematography was stunning with beautiful camera work and incredible shots. The soundtrack and sound effects perfectly complemented every tense scene."

print("\n" + "="*50)
print("DEMO: ASPECT SENTIMENT PREDICTIONS")
print("="*50)

predictions = predict(sample_review)
for aspect, info in predictions.items():
    print(f"\n[{aspect}]")
    print(f"  Sentiment  : {info['sentiment'].upper()}")
    print(f"  Confidence : {info['confidence']}")
    print(f"  Text used  : \"{info['text_used']}\"")

Dataset shape: (50000, 2)

Loading Sentence Transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading DistilBERT sentiment labeler...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

  Train Loss : 0.1012 | Train Acc : 96.42%
  Val   Acc  : 94.28%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved

FINAL TEST SET EVALUATION


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy : 94.28%

Classification Report:
              precision    recall  f1-score   support

    negative       0.95      0.92      0.94      2188
    positive       0.94      0.96      0.95      2658

    accuracy                           0.94      4846
   macro avg       0.94      0.94      0.94      4846
weighted avg       0.94      0.94      0.94      4846


DEMO: ASPECT SENTIMENT PREDICTIONS

[Acting performance of actors]
  Sentiment  : POSITIVE
  Confidence : 99.9%
  Text used  : "the acting was absolutely phenomenal, especially the lead actress..."

[Story and plot quality]
  Sentiment  : NEGATIVE
  Confidence : 100.0%
  Text used  : "the plot was confusing and hard to follow at times..."

[Cinematography and visuals]
  Sentiment  : POSITIVE
  Confidence : 100.0%
  Text used  : "the cinematography was stunning with beautiful camera work and incredible shots..."

[Background music and soundtrack]
  Sentiment  : POSITIVE
  Confidence : 99.9%
  Text used  : "the soundtrack an